# 02 · Explainable AI with SHAP
Explains the Random Forest saved by `01_flood_ml.ipynb`. Set `MODEL` to choose which one.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from src.config import DATASET_CSV, OUT, CAT
from src.ml_utils import add_features

MODEL = 'rf_no_elevation'          # or 'rf_all_features'
pipe = joblib.load(OUT / f'{MODEL}.pkl')
FEATURES = json.loads((OUT / 'model_features.json').read_text())[MODEL]
df = add_features(pd.read_csv(DATASET_CSV))
print(MODEL, '→', len(FEATURES), 'features')

In [ ]:
prep, model = pipe.named_steps['prep'], pipe.named_steps['model']
names = [n.replace('num__', '').replace('cat__', '') for n in prep.get_feature_names_out()]

# 400 points per event so every flood is represented
n_per_event = min(400, df['event'].value_counts().min())
sample = df.groupby('event').sample(n=n_per_event, random_state=42)
Xs = pd.DataFrame(prep.transform(sample[FEATURES]), columns=names, index=sample.index)

sv = shap.TreeExplainer(model).shap_values(Xs, check_additivity=False)
if isinstance(sv, list):
    sv = sv[1]
elif sv.ndim == 3:
    sv = sv[:, :, 1]               # contributions towards class 1 = flood

def group(n):                       # merge one-hot columns back into lulc / soil_tex
    for c in CAT:
        if n.startswith(c + '_'):
            return c
    return n

sv_grp = pd.DataFrame(sv, columns=names, index=sample.index).T.groupby(
    [group(n) for n in names]).sum().T
X_disp = sample[sv_grp.columns]     # original (unscaled) values for readable plots

In [ ]:
shap.summary_plot(sv_grp.values, X_disp, show=False)
plt.title(f'SHAP summary — {MODEL}'); plt.tight_layout()
plt.savefig(OUT / f'shap_summary_{MODEL}.png', dpi=200, bbox_inches='tight'); plt.show()

shap.summary_plot(sv_grp.values, X_disp, plot_type='bar', show=False)
plt.title('Mean |SHAP| — factor importance'); plt.tight_layout()
plt.savefig(OUT / f'shap_importance_{MODEL}.png', dpi=200, bbox_inches='tight'); plt.show()

In [ ]:
top4 = sv_grp.abs().mean().sort_values(ascending=False).index[:4]
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
for ax, f in zip(axes, top4):
    shap.dependence_plot(f, sv_grp.values, X_disp, interaction_index=None, ax=ax, show=False)
    ax.set_title(f)
plt.tight_layout(); plt.savefig(OUT / f'shap_dependence_top4_{MODEL}.png', dpi=200); plt.show()
print('Top 4 factors:', list(top4))

In [ ]:
# Do different floods have different drivers?
imp = sv_grp.abs().groupby(sample['event']).mean().T
imp = imp.loc[imp.mean(axis=1).sort_values().index]
imp.to_csv(OUT / f'shap_importance_by_event_{MODEL}.csv')
imp.plot.barh(figsize=(9, 7))
plt.xlabel('Mean |SHAP value|'); plt.title('Factor importance by flood event')
plt.tight_layout(); plt.savefig(OUT / f'shap_by_event_{MODEL}.png', dpi=200); plt.show()
imp.iloc[::-1].round(4)